In [ ]:
# This makes text wrap in the output box
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/NLP_Research_Project/"

In [14]:
import os
import pandas as pd
import torch
import pandas as pd
import csv


input_path = 'processed_data.csv'
df = pd.read_csv(input_path)

Load political bias classifier

In [ ]:
from smc_steer_summary import bias_model_factory

bias_model_path = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_path = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

In [27]:
def generate_summaries(tokenizer, model, df, out_path, is_gpt=False, is_llama=False, bias=False):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not is_llama:
      model.to(device)

    n_summaries = 3

    with open(out_path, 'w', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Title', 'Summary', 'Predicted Bias', 'Stance'])

    for idx, row in df.iterrows():
        title, text = row.title, row.body

        stances = ['left', 'right', 'center'] if bias else [row.stance]

        print(f'{idx}: {title}')

        for stance in stances:

            prompt = f'Summarize this article{" with " + stance + " bias" if bias else ""}: {text}'
            end_prompt = '\nSummary:'

            if hasattr(tokenizer, 'model_max_length'):
                summary_length = 512
                end_prompt_len = tokenizer(end_prompt, return_tensors="pt").input_ids.shape[1]
                max_length = tokenizer.model_max_length-summary_length-end_prompt_len if is_gpt or is_llama else summary_length
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length)
                prompt = tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True) + end_prompt
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length)
                prompt_len = inputs.input_ids.shape[1]
                offset = prompt_len

            else:
                prompt += end_prompt
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
                offset = 0


            print(prompt)
            print('------')

            inputs = inputs.to(device)

            summary_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                temperature = 0.8,
                min_length=3,
                max_new_tokens=512,
                do_sample=True,
                repetition_penalty=1.1,
                top_k=50,
                top_p=0.95,
                num_return_sequences=n_summaries,
                pad_token_id=tokenizer.eos_token_id 
            )

            for i, summary_id in enumerate(summary_ids):
                summary = tokenizer.decode(summary_id[offset:], skip_special_tokens=True)
                print('---')
                print(summary)

                pred_bias, _ = bias_model(summary)

                with open(out_path, 'a', encoding='utf-8') as f:
                  writer = csv.writer(f)
                  writer.writerow([title, summary, pred_bias, stance])

            del inputs
            del summary_ids

            print('------------------------------------------')

def summarize(tokenizer, model, df, out_path, is_gpt=False, is_llama=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt, is_llama, bias=False)

def summarize_with_leaning(tokenizer, model, df, out_path, is_gpt=False, is_llama=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt, is_llama, bias=True)

# BART

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

In [ ]:
summarize(bart_tokenizer, bart_model, df, 'bart.csv')

In [ ]:
summarize_with_leaning(bart_tokenizer, bart_model, df, 'bart-prompt.csv')

# T5

In [ ]:
from transformers import AutoTokenizer, AutoModelWithLMHead

t5_tokenizer = AutoTokenizer.from_pretrained('t5-base')
t5_model = AutoModelWithLMHead.from_pretrained('t5-base', return_dict=True)

In [ ]:
summarize(t5_tokenizer, t5_model, df, 't5.csv')

In [ ]:
summarize_with_leaning(t5_tokenizer, t5_model, df, 't5-prompt.csv')

# GPT2

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

In [ ]:
summarize(gpt2_tokenizer, gpt2_model, df, 'gpt2.csv', is_gpt=True)

In [ ]:
summarize_with_leaning(gpt2_tokenizer, gpt2_model, df, 'gpt2-prompt.csv', is_gpt=True)

# GPT-Neo

In [ ]:
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

neo_model = GPTNeoForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
neo_tokenizer = GPT2Tokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")

In [ ]:
summarize(neo_model, neo_tokenizer, df, 'neo.csv', is_gpt=True)

In [ ]:
summarize_with_leaning(neo_model, neo_tokenizer, df, 'neo-prompt.csv', is_gpt=True)

# Llama 2

In [ ]:
# need to login to use llama
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import LlamaForCausalLM, LlamaTokenizer

llama_tokenizer = LlamaTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
llama_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
llama_tokenizer.model_max_length = 2048

In [ ]:
summarize(llama_tokenizer, llama_model, df, 'llama.csv')

In [ ]:
summarize_with_leaning(llama_tokenizer, llama_model, df, 'llama-prompt.csv')